## Using PulseKit to run PTT pipeline

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PulseKit import *  
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import os

#### Load Dataset, Get Cardiac Cycles, and process signals

In [2]:
dataset = Dataset(
    datasetName="TestDataset",

    #optionally specify which sessions to include
    #sessionIds=["52", "53", "54", "55", "56", "57"], 
    
    signals=[
        ECG(name="ECG"),
        PPG(name="PPG_Finger"),
        PPG(name="PPG_Ear"),
        Stethoscope(name="Digital_Stethoscope"),
        Earbud(name="Earbud_Left"),
        Earbud(name="Earbud_Right"),
        Accelerometer(name="Accelerometer_X"),
        Accelerometer(name="Accelerometer_Y"),
        Accelerometer(name="Accelerometer_Z"),
        RespirationBelt(name="Respiration_Belt"),
        BP(name="Diastolic_BP"),
        BP(name="Systolic_BP"),
        BP(name="NIBP"),
        Hemodynamics(name="Cardiac_Output"),
        Hemodynamics(name="Stroke_Volume"),
        Hemodynamics(name="SVR"),
        Hemodynamics(name="Heart_Rate")
    ],
    data_directory="wfdb_data"
)

Skipping 102 RECORDS entries without a local WFDB header (for example: HC_11_1, HC_11_2, HC_12_1, HC_12_2, HC_13_1).


In [4]:
dataset.sessionIds

['HC_10_1', 'HC_10_2', 'HC_42_2', 'HFC_106_1', 'HFC_116_1']

In [5]:
dataset.get_cardiac_cycles(reference_signal_name="ECG")

Using segments file: segments_cardiac_cycles_reference_signal_name_ecg.csv


In [6]:
dataset.process_signals(signal_list=["ECG", "PPG_Finger", "PPG_Ear", "Earbud_Left", "Systolic_BP", "Diastolic_BP"])

Processing session HC_10_1...
Processing session HC_10_2...
Processing session HC_42_2...
Processing session HFC_106_1...
Processing session HFC_116_1...


#### Compute features and populate tables with targets

In [7]:
dataset.compute_features(
    features=[
        PWA(dataset=dataset, signals=["PPG_Finger", "PPG_Ear"]),
        PTT(dataset=dataset, signals=["PPG_Finger", "Earbud_Left", "ECG"])
    ],
    targets=["Systolic_BP", "Diastolic_BP"],
    output_name="test_features",
    activity_labels=True
)

Saved to: /Users/thomaskleinknecht/Desktop/BLOOD PRESSURE/pulsekit/wfdb_data/test_features.csv


#### Run Models: e.g. Linear PTT Model for Systolic BP

In [8]:
CALIBRATION_N = 8

# Load and clean data in one go
df = pd.read_csv("test_data/test_features.csv")
subset = df[["participant_id", "session_id", "PTT_ECG_PPG_Finger", "Systolic_BP_midpoint"]].dropna()

results = []

for subject_id, subject_data in subset.groupby("participant_id"):
    
    if len(subject_data) <= CALIBRATION_N:
        continue

    X = subject_data[["PTT_ECG_PPG_Finger"]] 
    y = subject_data["Systolic_BP_midpoint"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, train_size=CALIBRATION_N, random_state=42
    )

    # Train and predict
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Store results
    results.append({
        "subject_id": subject_id,
        "total_samples": len(subject_data),
        "MAE": round(mean_absolute_error(y_test, y_pred), 3),
        "R2": round(r2_score(y_test, y_pred), 3),
        "Slope": round(model.coef_[0], 4),
        "Intercept": round(model.intercept_, 4)
    })

final_df = pd.DataFrame(results)
print(final_df.head())

   subject_id  total_samples     MAE     R2   Slope  Intercept
0          53           1992   9.097  0.162 -0.7651   241.0126
1          54             20   7.795 -0.408 -0.0168   124.4635
2          55           1607   6.310  0.174 -0.1824   150.1948
3          56           1416   6.156  0.315 -0.3075   172.8337
4          57           1405  20.791 -3.551 -0.8016   293.1244
